# picoNewton v3 — mechanosensory observability and WSS nonredundancy

**Purpose.** This notebook extends the published anisotropic Womersley/Lamb-force study into a focused theoretical test of mechanosensory observability. It asks whether the published endothelial-scale hydrodynamic signal remains distinguishable from wall shear stress after kinetic filtering.

The notebook produces six nonredundant, multi-panel figures:

1. published-signal continuity and physical decomposition;
2. hydrodynamic WSS–Lamb nonredundancy before any sensor model;
3. analytical and nonlinear mechanosensory transfer functions;
4. anisotropy and harmonic source attribution;
5. leave-one-artery-out WSS-surrogate competition;
6. uncertainty-aware gate margins and retained-claim map.

> **Claim boundary.** The two-state sensor is a constitutive observability model. It does not identify a receptor, equate the Lamb vector with exact wall traction, or establish endothelial activation experimentally.

Primary hydrodynamic reference: Saqr KM, *Scientific Reports* 16, 12584 (2026), DOI `10.1038/s41598-026-47474-x`.


In [ ]:
import os

PROFILE = os.environ.get("PICONEWTON_PROFILE", "quick")
REPOSITORY_REF = os.environ.get("PICONEWTON_REF", "main")
STORAGE_MODE_OVERRIDE = os.environ.get("PICONEWTON_STORAGE")
DEVELOPMENT_SKIP_V2_HASH = os.environ.get("PICONEWTON_DEV_SKIP_V2_HASH", "0") == "1"

assert PROFILE in {"quick", "publication"}
print({"profile": PROFILE, "repository_ref": REPOSITORY_REF})


## 1. Environment, repository, and deterministic run

The notebook runs inside a repository checkout, in local Jupyter, or in Google Colab. The publication profile uses the locked numerical resolution and enforces the committed `picoNewton_v2.ipynb` identity before analysis.


In [ ]:
from __future__ import annotations

import json
import math
import shutil
import subprocess
import sys
from dataclasses import asdict
from pathlib import Path


def locate_project_root() -> Path | None:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if candidate.name == "picoNewton_v3" and (candidate / "src/piconewton_v3").is_dir():
            return candidate.resolve()
        nested = candidate / "picoNewton_v3"
        if (nested / "src/piconewton_v3").is_dir():
            return nested.resolve()
    return None


PROJECT_ROOT = locate_project_root()
if PROJECT_ROOT is None and "google.colab" in sys.modules:
    checkout = Path("/content/picoNewton")
    if not checkout.exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1", "--branch", REPOSITORY_REF,
                "https://github.com/khalid-saqr/picoNewton.git", str(checkout),
            ],
            check=True,
        )
    PROJECT_ROOT = checkout / "picoNewton_v3"
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate picoNewton_v3")

REPO_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print({"project_root": str(PROJECT_ROOT), "repo_root": str(REPO_ROOT)})


## 2. Imports, profiles, storage, and immutable thresholds

The `quick` profile validates the end-to-end notebook. The `publication` profile uses $N=150$, 2,048 time points per cardiac cycle, 256 near-wall quadrature points, and the complete sensor grid.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from piconewton_v3.model import (
    FluidProperties, HydrodynamicConfig, SensorConfig, V2_ARTERY_CASES,
    V2_EXPECTED_BLOB_SHA, compute_hydrodynamics, isotropic_validation,
    lamb_work, periodic_sensor_solution, rms_difference, signal_metrics, wss_work,
)
from piconewton_v3.provenance import (
    environment_snapshot, execute_stripped_v2, git_commit_or_unknown,
    strip_notebook_outputs, validate_v2_blob, write_json,
)
from piconewton_v3.study_io import StudyStore, resolve_study_root
from piconewton_v3.workflow import evaluate_effect_gates, fit_wss_surrogate, run_parameter_grid

CONFIG_PATH = PROJECT_ROOT / "configs" / f"{PROFILE}.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
GATES = json.loads((PROJECT_ROOT / "configs/effect_gates.json").read_text())
if STORAGE_MODE_OVERRIDE:
    CONFIG["storage"]["mode"] = STORAGE_MODE_OVERRIDE
OBSERVABILITY_CONFIG = {**CONFIG, "analysis": "mechanosensory_observability", "notebook_schema": "1.0.0", "surrogates": ["instantaneous", "lagged", "lagged_lowpass"]}
OUTPUT_ROOT, STORAGE_MODE = resolve_study_root(mode=CONFIG["storage"]["mode"], drive_subdir=CONFIG["storage"]["drive_subdir"], local_root=CONFIG["storage"]["local_root"])
STORE = StudyStore(OUTPUT_ROOT)
STORE.initialize_layout()
RUN_ID, RUN_ROOT = STORE.create_run(OBSERVABILITY_CONFIG, git_commit_or_unknown(REPO_ROOT), V2_EXPECTED_BLOB_SHA, CONFIG["solver_mode"], CONFIG["random_seed"])
STORE.set_status(RUN_ID, "running")
write_json(RUN_ROOT / "provenance/environment.json", environment_snapshot())
FIGURE_DIR = RUN_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print({"run_id": RUN_ID, "run_root": str(RUN_ROOT), "storage": STORAGE_MODE})


## 3. Published-ground-truth provenance guard

The original paper established the anisotropic Womersley solution, the endothelial-scale control-volume force, vessel-specific spectra, and a theoretical basis for future mechanobiology. This notebook does not re-claim those findings; it verifies continuity and tests their observability under sensor kinetics.


In [ ]:
V2_PATH = REPO_ROOT / "picoNewton_v2.ipynb"
if V2_PATH.exists():
    write_json(RUN_ROOT / "provenance/v2_hash_guard.json", validate_v2_blob(V2_PATH))
    stripped = RUN_ROOT / "provenance/picoNewton_v2_stripped.ipynb"
    write_json(RUN_ROOT / "provenance/v2_stripping.json", strip_notebook_outputs(V2_PATH, stripped))
    if CONFIG.get("run_v2_cold_regression"):
        write_json(RUN_ROOT / "provenance/v2_cold_execution.json", execute_stripped_v2(stripped, RUN_ROOT / "provenance/picoNewton_v2_executed.ipynb", timeout_s=CONFIG.get("v2_execution_timeout_s", 1800)))
elif PROFILE == "publication" or not DEVELOPMENT_SKIP_V2_HASH:
    raise FileNotFoundError(V2_PATH)
else:
    display(Markdown("**Development-only:** v2 hash guard skipped outside the parent repository."))


## 4. Mechanics and observables

The Gromeka–Lamb identity is

$$(\mathbf u\cdot\nabla)\mathbf u=\nabla(|\mathbf u|^2/2)-\mathbf u\times\boldsymbol\omega.$$

For $\mathbf u=(0,u_\theta,u_z)$, the actual radial convective acceleration is $-u_\theta^2/r$. The notebook therefore reports separately:

- signed control-volume Lamb force;
- magnitude-type exposure force used in the published analysis;
- isotropic Lamb baseline;
- anisotropy-excess force;
- the radial Gromeka–Lamb identity residual.


In [ ]:
SECTION = PROJECT_ROOT / "src/piconewton_v3/observability_notebook/section_01_hydrodynamic_decomposition.py"
exec(compile(SECTION.read_text(encoding="utf-8"), str(SECTION), "exec"), globals())

## Figure 1 — Published signal continuity and physical decomposition

This figure establishes continuity with the first paper while separating the observables required by the new study. The reproduction-layout comparison is a traceability diagnostic; all mechanosensory conclusions use the verified path.


In [ ]:
SECTION = PROJECT_ROOT / "src/piconewton_v3/observability_notebook/section_02_figure1_signal_continuity.py"
exec(compile(SECTION.read_text(encoding="utf-8"), str(SECTION), "exec"), globals())

## 5. Hydrodynamic nonredundancy before the sensor

A mechanosensor cannot create hydrodynamic information that is absent from its inputs. This section therefore challenges the signed Lamb-force waveform with three predefined WSS surrogates before introducing molecular coupling:

1. instantaneous affine WSS;
2. affine WSS with a periodic lag;
3. affine WSS with a periodic lag and first-order low-pass filtering.

The residual fraction is

$$R_\perp=\|F_L-\widehat F_L[\tau_w]\|_2/\|F_L-\overline{F_L}\|_2.$$


In [ ]:
SECTION = PROJECT_ROOT / "src/piconewton_v3/observability_notebook/section_03_hydrodynamic_nonredundancy.py"
exec(compile(SECTION.read_text(encoding="utf-8"), str(SECTION), "exec"), globals())

## 6. Analytical and nonlinear mechanosensory transfer

For weak input work $\Psi(t)$, the two-state model linearizes to

$$\frac{d\,\delta p}{dt}=-\frac{\delta p}{\tau_0}+\frac{p_0(1-p_0)}{\tau_0}\Psi(t).$$

For harmonic $h$ and $\Omega=\omega_0\tau_0$,

$$|\delta p_h|=\frac{p_0(1-p_0)|\Psi_h|}{\sqrt{1+(h\Omega)^2}}.$$

This section verifies the linear limit against the exact periodic nonlinear solution and maps the transition from weak response to nonlinear saturation.


In [ ]:
SECTION = PROJECT_ROOT / "src/piconewton_v3/observability_notebook/section_04_mechanosensory_transfer.py"
exec(compile(SECTION.read_text(encoding="utf-8"), str(SECTION), "exec"), globals())

## 7. Source attribution: anisotropy and higher harmonics

The first paper established a hydrodynamic anisotropy-induced spectral signature. The new question is whether that source survives kinetic filtering. Ablation effects are evaluated with identical sensor parameters:

- anisotropy effect: full anisotropic signal versus isotropic baseline;
- high-harmonic effect: six-harmonic signal versus $h\leq2$;
- direction effect: signed sensitivity versus reversed sensitivity.


In [ ]:
SECTION = PROJECT_ROOT / "src/piconewton_v3/observability_notebook/section_05_source_attribution.py"
exec(compile(SECTION.read_text(encoding="utf-8"), str(SECTION), "exec"), globals())

## 8. Leave-one-artery-out WSS-surrogate competition

This is the decisive nonredundancy challenge. A global lag and, where applicable, a first-order filter are fitted on five arteries. In the held-out artery, the WSS surrogate is deliberately mean- and RMS-matched to the Lamb-force waveform. The challenge therefore tests waveform information rather than amplitude alone.

Three fixed model classes are evaluated without adding complexity after observing the result.


In [ ]:
SECTION = PROJECT_ROOT / "src/piconewton_v3/observability_notebook/section_06_loo_wss_competition.py"
exec(compile(SECTION.read_text(encoding="utf-8"), str(SECTION), "exec"), globals())

## 9. Robustness, gate margins, and retained claims

Binary gates are retained for governance, but the main figure reports each criterion as a margin relative to its threshold. A margin above one passes; a margin below one fails. This prevents visually exaggerating near-threshold outcomes.


In [ ]:
SECTION = PROJECT_ROOT / "src/piconewton_v3/observability_notebook/section_07_gate_margins.py"
exec(compile(SECTION.read_text(encoding="utf-8"), str(SECTION), "exec"), globals())

## 10. Publication tables, figure source data, and reproducibility bundle

Every plotted quantity is exported as tidy CSV data. The notebook also records the exact configuration, environment, v2 identity, WSS-surrogate parameters, figure manifest, and checksums.


In [ ]:
SECTION = PROJECT_ROOT / "src/piconewton_v3/observability_notebook/section_08_export.py"
exec(compile(SECTION.read_text(encoding="utf-8"), str(SECTION), "exec"), globals())

## 11. Interpretation contract

The notebook is intentionally outcome-neutral.

- A robust residual after the strongest held-out WSS challenge supports nonredundant mechanosensory information.
- A connected but narrow region supports a conditional observability claim.
- Failure after kinetic filtering is a scientifically useful negative result: the published hydrodynamic signal can be real while remaining biologically unresolved under the modeled kinetics.
- Anisotropy-specific and high-harmonic claims are retained only if their independent ablation gates pass.

The final manuscript should use the six multi-panel figures above as the main result sequence and move numerical convergence, full control tables, and complete artery waveforms to supplementary material.


In [ ]:
for figure in [FIG1, FIG2, FIG3, FIG4, FIG5, FIG6]:
    display(Image(filename=str(figure), width=900))

display(GATE_MARGIN_TABLE)
